In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from pandas.conftest import axis


In [ ]:
import pandas as pd
aus_tourism = pd.read_csv('data/aus_tourism.csv', sep=',', na_values=[''], quotechar='"')
aus_tourism["unique_id"].unique()

In [ ]:
mean_aus_tourism = aus_tourism.groupby("unique_id",as_index=False)["y"].mean().round(2)
mean_aus_tourism.sort_values(by="y").head()

In [ ]:
aus_tourism.describe()

In [ ]:
aus_tourism.dtypes
aus_tourism["ds"] = pd.to_datetime(aus_tourism["ds"])

In [ ]:
from tsfeatures import tsfeatures, statistics

summary_stats = tsfeatures(
    aus_tourism,
    freq=4,                     # 季度数据，频率为 4
    features=[statistics],      # 你要提取的特征列表
    scale=False,                # 不进行缩放
)

In [ ]:
summary_stats1 = tsfeatures(
    aus_tourism,
    freq=4,                     # 季度数据，频率为 4
    features=[statistics],      # 你要提取的特征列表
    scale=False,                # 不进行缩放
)

In [ ]:
summary_stats1.head()

In [ ]:
import tsfeatures as tsf

# 先确保 ds 是 datetime，避免频率推断异常
aus_tourism["ds"] = pd.to_datetime(aus_tourism["ds"])

summary_stats2 = tsf.tsfeatures(
    aus_tourism,
    freq=4, features=[tsf.acf_features], scale=False,
)
summary_stats2.head()


In [ ]:
import tsfeatures as tsf


aus_tourism["ds"] = pd.to_datetime(aus_tourism["ds"])

stl_feat = tsf.tsfeatures(
    aus_tourism,
    freq=4, features=[tsf.stl_features], scale=False,
)
stl_feat


In [ ]:
aus_tourism.head()
stl_feat["unique_id"].unique()

In [ ]:
df = stl_feat["unique_id"].str.split("-",expand=True).rename(columns={0: "region", 1: "state", 2: "purpose"}).join(stl_feat)

df

In [ ]:
fig, axis = plt.subplots(3, 3, figsize=(10, 10))
axes = axis.flatten()

for ax, (state, state_df) in zip(axes, df.groupby("state")):
    for purpose, purpose_df in state_df.groupby("purpose"):
        ax.scatter(
            purpose_df["trend"],
            purpose_df["seasonal_strength"],
            label=purpose,
            alpha=0.8,
        )
    ax.set_title(state)
    ax.set_xlabel("trend")
    ax.set_ylabel("seasonal_strength")

# 隐藏多余子图（如果 state 少于 9 个）
for ax in axes[len(df["state"].unique()):]:
    ax.set_visible(False)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    title="Purpose",
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
)
fig.tight_layout(rect=[0, 0, 0.88, 1])


In [ ]:
stl_feat

In [ ]:
row = stl_feat.loc[lambda x:x["seasonal_strength"].idxmax()]

uid = row["unique_id"]
df_plot = aus_tourism[aus_tourism["unique_id"] == uid]
df_plot.head()

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(df_plot["ds"], df_plot["y"])
plt.show()

In [ ]:
all_features = [
    tsf.acf_features,
    tsf.arch_stat,
    tsf.crossing_points,
    tsf.entropy,
    tsf.flat_spots,
    tsf.heterogeneity,
    tsf.holt_parameters,
    tsf.lumpiness,
    tsf.nonlinearity,
    tsf.pacf_features,
    tsf.stl_features,
    tsf.stability,
    tsf.hw_parameters,
    tsf.unitroot_kpss,
    tsf.unitroot_pp,
    tsf.series_length,
    tsf.hurst,
]
all_feat = tsf.tsfeatures(aus_tourism, freq=4, features=all_features)
all_feat.head(10)


In [ ]:
seasonal_feat = all_feat[
    ["unique_id", "seasonal_strength", "peak", "trough",
     "seas_acf1", "seas_pacf"]
]
df = (
    seasonal_feat["unique_id"].str.split("-", expand=True)
    .rename(columns={0: "region", 1: "state", 2: "Purpose"})
    .join(seasonal_feat)
)
g = sns.pairplot(df, hue="Purpose")
g.fig.set_size_inches(10, 10)
plt.show()

In [ ]:
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
df = (
    all_feat
    .assign(purpose=lambda x: x["unique_id"].str.split("-").str[-1])
    .dropna(axis='columns')
)
X = df.drop(columns=["unique_id", "purpose"])

pipeline = Pipeline([
    ('scale', StandardScaler()),
    ('pca', PCA(n_components=2)),
])

principal_components = pipeline.fit_transform(X)
pca_df = (
    pd.DataFrame(data=principal_components, columns=["PC1", "PC2"])
    .join(df[["unique_id", "purpose"]])
)
fig, ax = plt.subplots()
sns.scatterplot(x="PC1", y="PC2", s=80,
    hue="purpose", edgecolor="none", data=pca_df, ax=ax)
ax.set(
    title="PCA of Features",
    xlabel="Principal Component 1",
    ylabel="Principal Component 2",
)
ax.get_legend().remove()
handles, labels = ax.get_legend_handles_labels()
fig.legend(handles, labels,
    title="Purpose", loc="center left", frameon=False,
    bbox_to_anchor=(1.02, .5), borderaxespad=0)
plt.show()

In [ ]:
from sklearn.neighbors import LocalOutlierFactor
lof = LocalOutlierFactor(n_neighbors=10).fit(pca_df[["PC1", "PC2"]])
df = pca_df.assign(score=lof.negative_outlier_factor_)
select_id = df.sort_values(by="score").head(5)["unique_id"]
fig, axis = plt.subplots(5,figsize=(8, 8),sharex=True)
for (ax,select_id) in zip(axis, select_id):
    df = aus_tourism.loc[lambda x:x["unique_id"] == select_id]
    ax.plot(df["ds"], df["y"], color="k")
    ax.set_title(unique_id, loc="left", size="medium")

fig.suptitle("Outlying time series in PC space")
fig.supylabel("Trips")
fig.supxlabel("Quarter")
plt.show()

In [ ]:
lof = LocalOutlierFactor(n_neighbors=10).fit(pca_df[["PC1", "PC2"]])
res = pca_df.assign(score=lof.negative_outlier_factor_)
selected_ids = res.sort_values("score")["unique_id"].head(3)

fig, axs = plt.subplots(3, figsize=(8, 8), sharex=True)
for (ax, unique_id) in zip(axs, selected_ids):
    df = aus_tourism.loc[lambda x: x["unique_id"] == unique_id]
    ax.plot(df["ds"], df["y"], color="k")
    ax.set_title(unique_id, loc="left", size="medium")

fig.suptitle("Outlying time series in PC space")
fig.supylabel("Trips")
fig.supxlabel("Quarter")
plt.show()